In [1]:
import pandas as pd
from pathlib import Path

flights = pd.read_csv("data/raw/flights.csv")
weather = pd.read_csv("data/raw/weather.csv")
airlines = pd.read_csv("data/raw/airlines.csv")

print("Flights:", flights.shape)
print("Weather:", weather.shape)

Flights: (336776, 19)
Weather: (26115, 15)


In [2]:
season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall"
}

flights["season"] = flights["month"].map(season_map)

print(flights["season"].value_counts())

season
Summer    86995
Spring    85960
Fall      83731
Winter    80090
Name: count, dtype: int64


In [3]:
flights["scheduled_hour"] = (
    flights["sched_dep_time"] // 100
).astype(int)

print(flights["scheduled_hour"].value_counts().sort_index())

print(
    "Invalid scheduled hours:",
    (~flights["scheduled_hour"].between(0, 23)).sum()
)

scheduled_hour
1         1
5      1953
6     25951
7     22821
8     27242
9     20312
10    16708
11    16033
12    18181
13    19956
14    21706
15    23888
16    23002
17    24426
18    21783
19    21441
20    16739
21    10933
22     2639
23     1061
Name: count, dtype: int64
Invalid scheduled hours: 0


In [4]:
flights["time_hour"] = pd.to_datetime(
    flights["time_hour"], utc=True
)

weather["time_hour"] = pd.to_datetime(
    weather["time_hour"], utc=True
)

merged = flights.merge(
    weather,
    on=["origin", "time_hour"],
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("", "_weather")
)

print("Flights before merge:", len(flights))
print("Flights after merge:", len(merged))

print("\nWeather merge status:")
print(merged["_merge"].value_counts())

assert len(merged) == len(flights), "Unexpected row count change!"

Flights before merge: 336776
Flights after merge: 336776

Weather merge status:
_merge
both          335220
left_only       1556
right_only         0
Name: count, dtype: int64


In [5]:
analysis = merged.loc[merged["arr_delay"].notna()].copy()

print("Original flights:", len(flights))
print("Missing arrival delay:", merged["arr_delay"].isna().sum())
print("Analysis sample:", len(analysis))

print("\nWeather merge status in analysis sample:")
print(analysis["_merge"].value_counts())

Original flights: 336776
Missing arrival delay: 9430
Analysis sample: 327346

Weather merge status in analysis sample:
_merge
both          325819
left_only       1527
right_only         0
Name: count, dtype: int64


In [6]:
weather_vars = [
    "temp", "dewp", "humid", "wind_speed",
    "precip", "visib"
]

weather_missing = pd.DataFrame({
    "Missing Count": analysis[weather_vars].isna().sum(),
    "Missing %": (
        analysis[weather_vars].isna().mean() * 100
    ).round(2)
})

display(weather_missing)

,Missing Count,Missing %
temp,1544,0.47
dewp,1544,0.47
humid,1544,0.47
wind_speed,1605,0.49
precip,1527,0.47
visib,1527,0.47


In [7]:
print("Analysis rows:", len(analysis))

print("Missing season:", analysis["season"].isna().sum())

print(
    "Invalid scheduled hours:",
    (~analysis["scheduled_hour"].between(0, 23)).sum()
)

print(
    "Missing flight distance:",
    analysis["distance"].isna().sum()
)

print(
    "Missing arrival delay:",
    analysis["arr_delay"].isna().sum()
)

print(
    "Unique airports:",
    sorted(analysis["origin"].unique())
)

Analysis rows: 327346
Missing season: 0
Invalid scheduled hours: 0
Missing flight distance: 0
Missing arrival delay: 0
Unique airports: ['EWR', 'JFK', 'LGA']


In [8]:
from pathlib import Path

clean_dir = Path("data/cleaned")
clean_dir.mkdir(parents=True, exist_ok=True)

# Keep all flights with a valid arrival delay.
# Weather missing values are retained for now.
analysis_clean = analysis.drop(columns=["_merge"]).copy()

output_path = clean_dir / "analysis_ready_flights.csv"
analysis_clean.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(analysis_clean))
print("Columns:", len(analysis_clean.columns))

assert len(analysis_clean) == 327346
assert analysis_clean["arr_delay"].notna().all()

print("Validation passed!")

Saved: data/cleaned/analysis_ready_flights.csv
Rows: 327346
Columns: 34
Validation passed!
